# The Annotated Transformer — German → English Translation

This notebook documents the actual code in this repository, and is written so it can be run top-to-bottom as
a single script: each cell only depends on things already defined in an earlier cell — the same order
`transformer.py`, `helper.py`, and `translation.py` build things in in the real project (leaf modules first,
then whatever's composed out of them). Every code cell below is this project's real implementation.

In this notebook you'll learn:

- What a transformer is, and how this repo's encoder/decoder/attention modules are built, bottom-up: building
  blocks (embeddings, positional encoding, feed-forward, multi-head attention) → layers → stacks → the
  top-level model.
- How this repo loads and batches its German→English parallel data (`data/train.de`, `data/train.en`, ...).
- How training and greedy-decoding translation work here, end to end.

This project does **not** implement a few things you may have seen in other transformer walkthroughs — a
custom learning-rate warm-up schedule, label smoothing, BLEU scoring, dataset caching, automatic
pretrained-model download, or beam search. Those sections have been left out rather than kept as unused
reference material, so this notebook only describes what actually runs.

## What are transformers?

Transformers were originally proposed by Vaswani et al. in [Attention Is All You Need](https://arxiv.org/pdf/1706.03762.pdf).
The key idea is that you don't need recurrent or convolutional layers at all — attention alone, stacked into
an encoder/decoder architecture, is enough. That gives better long-range dependency modeling and, because
there's no sequential recurrence, the architecture is highly parallelizable.

---


In [2]:
# Imports actually used across this project's modules (transformer.py, helper.py, translation.py, main.py)

import math
import os
import time

import torch
import torch.nn as nn
from torch import Tensor
from torch.nn.functional import pad

from torch.utils.data import DataLoader
from torchtext.data.functional import to_map_style_dataset
from torchtext.vocab import build_vocab_from_iterator
import torchtext.datasets as datasets
import spacy

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from mpl_toolkits import mplot3d


c:\Users\chinh\.conda\envs\pytorch-transformer-de-to-en-translation\Lib\site-packages\torchtext\data\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
c:\Users\chinh\.conda\envs\pytorch-transformer-de-to-en-translation\Lib\site-packages\torchtext\vocab\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
c:\Users\chinh\.conda\envs\pytorch-transformer-de-to-en-t

In [3]:
# Hyperparameters as actually set in main.py (a much smaller model than the paper's base
# transformer, sized down for this project's Multi30k-style German/English corpus)

N_EPOCHS = 10
CLIP = 1
BATCH_SIZE = 128
MAX_PADDING = 20
LEARNING_RATE = 0.0005

N_LAYERS = 3       # paper default is 6; helper.make_model defaults to 3
D_MODEL = 256       # paper default is 512; main.py trains at 256
D_FFN = 512         # paper default is 2048; main.py trains at 512
N_HEADS = 8
DROPOUT = 0.1
MAX_LENGTH = 50     # max sequence length for positional encodings

# There's no CHECKPOINTS_PATH/BINARIES_PATH directory scheme here - the best model
# (by validation loss) is saved directly to a single file, 'transformer-model.pt', at the repo root.


# Part 1: The model

There are two important parts of any deep learning pipeline: the model and the data. We'll build the model
bottom-up here — the order Python actually needs to see these definitions in for the notebook to run without
`NameError`s: first the leaf building blocks (embeddings, positional encoding, feed-forward, multi-head
attention), then the encoder/decoder layers built from those, then the encoder/decoder stacks, and finally the
top-level `Transformer` that wires it all together.


## Building blocks: feed-forward, embeddings, positional encoding

These three modules don't depend on anything else in the model — they're the leaves of the dependency tree,
so they're defined first.


In [5]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ffn: int, dropout: float = 0.1):
        """
        Args:
            d_model:      dimension of embeddings
            d_ffn:        dimension of feed-forward network
            dropout:      probability of dropout occurring
        """
        super().__init__()

        self.w_1 = nn.Linear(d_model, d_ffn)
        self.w_2 = nn.Linear(d_ffn, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        Args:
            x:            output from attention (batch_size, seq_length, d_model)

        Returns:
            expanded-and-contracted representation (batch_size, seq_length, d_model)
        """
        # w_1(x).relu(): (batch_size, seq_length, d_model) x (d_model,d_ffn) -> (batch_size, seq_length, d_ffn)
        # w_2(w_1(x).relu()): (batch_size, seq_length, d_ffn) x (d_ffn, d_model) -> (batch_size, seq_length, d_model)
        return self.w_2(self.dropout(self.w_1(x).relu()))


class Embeddings(nn.Module):
    def __init__(self, vocab_size: int, d_model: int):
        """
        Args:
          vocab_size:     size of vocabulary
          d_model:        dimension of embeddings
        """
        super().__init__()

        # embedding look-up table (lut)
        self.lut = nn.Embedding(vocab_size, d_model)

        # dimension of embeddings
        self.d_model = d_model

    def forward(self, x: Tensor):
        """
        Args:
          x:              input Tensor (batch_size, seq_length)

        Returns:
                          embedding vector
        """
        # embeddings by constant sqrt(d_model)
        return self.lut(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_length: int = 5000):
        """
        Args:
          d_model:      dimension of embeddings
          dropout:      probability of dropout applied after adding the positional encodings (regularization)
          max_length:   maximum sequence length supported by the precomputed encoding table (default 5000)
        """
        super().__init__()

        self.dropout = nn.Dropout(p=dropout)

        # create tensor of 0s
        pe = torch.zeros(max_length, d_model)

        # create position column
        k = torch.arange(0, max_length).unsqueeze(1)

        # calc divisor for positional encoding
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )

        # calc sine on even indices
        pe[:, 0::2] = torch.sin(k * div_term)

        # calc cosine on odd indices
        pe[:, 1::2] = torch.cos(k * div_term)

        # add dimension
        pe = pe.unsqueeze(0)

        # buffers are saved in state_dict but not trained by the optimizer
        self.register_buffer("pe", pe)

    def forward(self, x: Tensor):
        """
        Args:
          x:        embeddings (batch_size, seq_length, d_model)

        Returns:
                    embeddings + positional encodings (batch_size, seq_length, d_model)
        """
        # add positional encoding to the embeddings
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)

        # perform dropout
        return self.dropout(x)


Each `EncoderLayer`/`DecoderLayer` sublayer (defined next) applies **post-norm**:
`LayerNorm(x + Dropout(Sublayer(x)))`, i.e. "Add & Norm" happens *after* the sublayer, exactly as in the
original paper — not the pre-norm variant (`x + Dropout(Sublayer(LayerNorm(x)))`) some later implementations
use. That logic is inlined directly in the `EncoderLayer`/`DecoderLayer` forward methods; there's no generic
`SublayerLogic` wrapper module.

## Multi-head attention

The last building block: the multi-head attention module used for encoder self-attention, decoder masked
self-attention, and decoder cross-attention alike (three separate instances, same class).


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int = 512, n_heads: int = 8, dropout: float = 0.1):
        """
        Args:
            d_model:      dimension of embeddings
            n_heads:      number of self attention heads
            dropout:      probability of dropout occurring
        """
        super().__init__()
        assert d_model % n_heads == 0  # ensure an even num of heads
        self.d_model = d_model  # 512 dim
        self.n_heads = n_heads  # 8 heads
        self.d_key = d_model // n_heads  # assume d_value equals d_key | 512/8=64

        self.Wq = nn.Linear(d_model, d_model)  # query weights
        self.Wk = nn.Linear(d_model, d_model)  # key weights
        self.Wv = nn.Linear(d_model, d_model)  # value weights
        self.Wo = nn.Linear(d_model, d_model)  # output weights

        self.dropout = nn.Dropout(p=dropout)  # initialize dropout layer

    def forward(self, query: Tensor, key: Tensor, value: Tensor, mask: Tensor = None):
        """
        Args:
           query:         query vector         (batch_size, q_length, d_model)
           key:           key vector           (batch_size, k_length, d_model)
           value:         value vector         (batch_size, s_length, d_model)
           mask:          optional padding/causal mask, broadcastable to (batch_size, n_heads, q_length, k_length);
                          used for encoder self-attention, decoder masked self-attention, and decoder cross-attention alike

        Returns:
           output:        attention values     (batch_size, q_length, d_model)
           attn_probs:    softmax scores       (batch_size, n_heads, q_length, k_length)
        """
        batch_size = key.size(0)

        # calculate query, key, and value tensors
        Q = self.Wq(query)
        K = self.Wk(key)
        V = self.Wv(value)

        # split each tensor into n-heads to compute attention
        Q = Q.view(batch_size, -1, self.n_heads, self.d_key).permute(0, 2, 1, 3)
        K = K.view(batch_size, -1, self.n_heads, self.d_key).permute(0, 2, 1, 3)
        V = V.view(batch_size, -1, self.n_heads, self.d_key).permute(0, 2, 1, 3)

        # computes attention
        # scaled dot product -> QK^{T}
        scaled_dot_prod = torch.matmul(Q, K.permute(0, 1, 3, 2)) / math.sqrt(self.d_key)

        # fill those positions with product as (-1e10) where mask positions are 0
        if mask is not None:
            scaled_dot_prod = scaled_dot_prod.masked_fill(mask == 0, -1e10)

        # apply softmax
        attn_probs = torch.softmax(scaled_dot_prod, dim=-1)

        # multiply by values to get attention
        A = torch.matmul(self.dropout(attn_probs), V)

        # reshape attention back to (batch_size, q_length, d_model)
        A = A.permute(0, 2, 1, 3).contiguous()
        A = A.view(batch_size, -1, self.n_heads * self.d_key)

        # push through the final weight layer
        output = self.Wo(A)

        return output, attn_probs  # return attn_probs for visualization of the scores


Two implementation details worth flagging against other reference implementations you may have seen:

- Masking uses `masked_fill(mask == 0, -1e10)` — a large negative finite number, not `-inf`. Functionally
  equivalent after softmax, but avoids any `NaN` risk if a row were ever fully masked.
- `Q`/`K`/`V` projections are three separate `nn.Linear` layers (`Wq`, `Wk`, `Wv`) rather than one fused
  `qkv` projection — simpler to read, at the cost of three small matmuls instead of one bigger one.

## Encoder

With the building blocks defined, we can compose them into the encoder layer and stack.

High level overview of the encoder:

1. It takes a batch of source sequences whose tokens have already been embedded (token embedding + positional encoding).
2. It runs `N_LAYERS` (3, by default here — the paper's base model uses 6) iterations of self-attention + feed-forward mixing.
3. The final output is consumed by the decoder's cross-attention.


In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ffn: int, dropout: float):
        """
        Args:
            d_model:      dimension of embeddings
            n_heads:      number of heads
            d_ffn:        dimension of feed-forward network
            dropout:      probability of dropout occurring
        """
        super().__init__()
        # multi-head attention sublayer
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        # layer norm for multi-head attention
        self.attn_layer_norm = nn.LayerNorm(d_model)

        # position-wise feed-forward network
        self.positionwise_ffn = PositionwiseFeedForward(d_model, d_ffn, dropout)
        # layer norm for position-wise ffn
        self.ffn_layer_norm = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src: Tensor, src_mask: Tensor):
        """
        Args:
            src:          positionally embedded sequences   (batch_size, seq_length, d_model)
            src_mask:     mask for the sequences            (batch_size, 1, 1, seq_length)
        Returns:
            src:          sequences after self-attention    (batch_size, seq_length, d_model)
        """
        # pass embeddings through multi-head attention
        _src, attn_probs = self.attention(src, src, src, src_mask)

        # residual add and norm
        src = self.attn_layer_norm(src + self.dropout(_src))

        # position-wise feed-forward network
        _src = self.positionwise_ffn(src)

        # residual add and norm
        src = self.ffn_layer_norm(src + self.dropout(_src))

        return src, attn_probs


class Encoder(nn.Module):
    def __init__(self, d_model: int, n_layers: int,
                 n_heads: int, d_ffn: int, dropout: float = 0.1):
        """
        Args:
            d_model:      dimension of embeddings
            n_layers:     number of encoder layers
            n_heads:      number of heads
            d_ffn:        dimension of feed-forward network
            dropout:      probability of dropout occurring
        """
        super().__init__()

        # create n_layers encoders
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ffn, dropout)
                                     for layer in range(n_layers)])

        self.dropout = nn.Dropout(dropout)

    def forward(self, src: Tensor, src_mask: Tensor):
        """
        Args:
            src:          embedded sequences                (batch_size, seq_length, d_model)
            src_mask:     mask for the sequences            (batch_size, 1, 1, seq_length)

        Returns:
            src:          sequences after self-attention    (batch_size, seq_length, d_model)
        """

        # pass the sequences through each encoder
        for layer in self.layers:
            src, attn_probs = layer(src, src_mask)

        self.attn_probs = attn_probs

        return src


Note the last line of `Encoder.forward`: `self.attn_probs` is overwritten on every loop iteration, so after
the forward pass it only holds the **last** layer's attention weights, not every layer's. That matters later
when visualizing attention (Part 5) — you can only inspect the final encoder/decoder layer here, not each
layer independently.

Also, unlike an approach that deep-copies one shared `EncoderLayer`/`MultiHeadAttention` module N times,
`Encoder` and `DecoderLayer` (next) each construct genuinely independent instances directly (a list
comprehension for the stack, and two separate `MultiHeadAttention` instances inside `DecoderLayer` for self-
vs. cross-attention) — there's no shared base module being cloned.

## Decoder

The decoder has a similar structure to the encoder:

1. It starts with a batch of target sequences whose tokens have already been embedded.
2. It does `N_LAYERS` iterations of masked self-attention, then cross-attention over the encoder's output.
3. The final decoder representations are projected straight to target-vocabulary logits.

The decoder uses a **causal mask** so tokens can't attend to future positions — built via `torch.tril` inside
`Transformer.make_trg_mask` (below) and combined with the padding mask.


In [ ]:
class DecoderLayer(nn.Module):

    def __init__(self, d_model: int, n_heads: int, d_ffn: int, dropout: float):
        """
        Args:
            d_model:      dimension of embeddings
            n_heads:      number of heads
            d_ffn:        dimension of feed-forward network
            dropout:      probability of dropout occurring
        """
        super().__init__()
        # masked multi-head attention sublayer
        self.masked_attention = MultiHeadAttention(d_model, n_heads, dropout)
        # layer norm for masked multi-head attention
        self.masked_attn_layer_norm = nn.LayerNorm(d_model)

        # multi-head attention sublayer
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        # layer norm for multi-head attention
        self.attn_layer_norm = nn.LayerNorm(d_model)

        # position-wise feed-forward network
        self.positionwise_ffn = PositionwiseFeedForward(d_model, d_ffn, dropout)
        # layer norm for position-wise ffn
        self.ffn_layer_norm = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, trg: Tensor, src: Tensor, trg_mask: Tensor, src_mask: Tensor):
        """
        Args:
            trg:          embedded sequences                (batch_size, trg_seq_length, d_model)
            src:          embedded sequences                (batch_size, src_seq_length, d_model)
            trg_mask:     mask for the sequences            (batch_size, 1, trg_seq_length, trg_seq_length)
            src_mask:     mask for the sequences            (batch_size, 1, 1, src_seq_length)

        Returns:
            trg:                sequences after self-attention    (batch_size, trg_seq_length, d_model)
            attn_probs:         cross-attention softmax scores    (batch_size, n_heads, trg_seq_length, src_seq_length)
            masked_attn_probs:  self-attention softmax scores     (batch_size, n_heads, trg_seq_length, trg_seq_length)
        """
        # pass trg embeddings through masked multi-head attention
        _trg, masked_attn_probs = self.masked_attention(trg, trg, trg, trg_mask)

        # residual add and norm
        trg = self.masked_attn_layer_norm(trg + self.dropout(_trg))

        # pass trg and src embeddings through multi-head attention
        _trg, attn_probs = self.attention(trg, src, src, src_mask)

        # residual add and norm
        trg = self.attn_layer_norm(trg + self.dropout(_trg))

        # position-wise feed-forward network
        _trg = self.positionwise_ffn(trg)

        # residual add and norm
        trg = self.ffn_layer_norm(trg + self.dropout(_trg))

        return trg, attn_probs, masked_attn_probs


class Decoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_layers: int,
                 n_heads: int, d_ffn: int, dropout: float = 0.1):
        """
        Args:
            vocab_size:   size of the target vocabulary
            d_model:      dimension of embeddings
            n_layers:     number of encoder layers
            n_heads:      number of heads
            d_ffn:        dimension of feed-forward network
            dropout:      probability of dropout occurring
        """
        super().__init__()

        # create n_layers encoders
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ffn, dropout)
                                     for layer in range(n_layers)])

        self.dropout = nn.Dropout(dropout)

        # set output layer
        self.Wo = nn.Linear(d_model, vocab_size)

    def forward(self, trg: Tensor, src: Tensor, trg_mask: Tensor, src_mask: Tensor):
        """
        Args:
            trg:          embedded sequences                (batch_size, trg_seq_length, d_model)
            src:          encoded sequences from encoder    (batch_size, src_seq_length, d_model)
            trg_mask:     mask for the sequences            (batch_size, 1, trg_seq_length, trg_seq_length)
            src_mask:     mask for the sequences            (batch_size, 1, 1, src_seq_length)

        Returns:
            output:             sequences after decoder           (batch_size, trg_seq_length, vocab_size)
            attn_probs:         cross-attention softmax scores    (batch_size, n_heads, trg_seq_length, src_seq_length)
            masked_attn_probs:  self-attention softmax scores     (batch_size, n_heads, trg_seq_length, trg_seq_length)
        """

        # pass the sequences through each decoder
        for layer in self.layers:
            trg, attn_probs, masked_attn_probs = layer(trg, src, trg_mask, src_mask)

        self.attn_probs = attn_probs
        self.masked_attn_probs = masked_attn_probs

        return self.Wo(trg)


There's no separate `DecoderGenerator` module here. `Decoder.Wo` — a single `nn.Linear(d_model, vocab_size)` —
projects the final decoder representations straight to vocabulary-size **logits**. There's no explicit
log-softmax call, because training (Part 3) uses `nn.CrossEntropyLoss`, which applies log-softmax internally.

## The top-level `Transformer`

Now that `Encoder`, `Decoder`, and `Embeddings` all exist, we can define the top-level module that wires them
together — the same order `helper.make_model` (Part 3) constructs them in.


In [ ]:
class Transformer(nn.Module):
    def __init__(
            self,
            encoder: Encoder,
            decoder: Decoder,
            src_embed: Embeddings,
            trg_embed: Embeddings,
            src_pad_idx: int,
            trg_pad_idx: int, device):
        """
        Args:
            encoder:      encoder stack
            decoder:      decoder stack
            src_embed:    source embeddings and encodings
            trg_embed:    target embeddings and encodings
            src_pad_idx:  padding index
            trg_pad_idx:  padding index
            device:       cuda or cpu
        """
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.trg_embed = trg_embed
        self.device = device
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx

    def make_src_mask(self, src: Tensor):
        """
        Args:
            src:          raw sequences with padding        (batch_size, seq_length)

        Returns:
            src_mask:     mask for each sequence            (batch_size, 1, 1, seq_length)
        """
        # assign 1 to tokens that need attended to and 0 to padding tokens, then add 2 dimensions
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)

        return src_mask

    def make_trg_mask(self, trg: Tensor):
        """
        Args:
            trg:          raw sequences with padding        (batch_size, seq_length)

        Returns:
            trg_mask:     mask for each sequence            (batch_size, 1, seq_length, seq_length)
        """

        seq_length = trg.shape[1]

        # assign True to tokens that need attended to and False to padding tokens, then add 2 dimensions
        trg_mask = (trg != self.trg_pad_idx).unsqueeze(1).unsqueeze(2)  # (batch_size, 1, 1, seq_length)

        # generate subsequent mask
        trg_sub_mask = torch.tril(
            torch.ones((seq_length, seq_length), device=self.device)).bool()  # (batch_size, 1, seq_length, seq_length)

        # bitwise "and" operator | 0 & 0 = 0, 1 & 1 = 1, 1 & 0 = 0
        trg_mask = trg_mask & trg_sub_mask

        return trg_mask

    def forward(self, src: Tensor, trg: Tensor):
        """
        Args:
            trg:          raw target sequences              (batch_size, trg_seq_length)
            src:          raw src sequences                 (batch_size, src_seq_length)

        Returns:
            output:       sequences after decoder           (batch_size, trg_seq_length, output_dim)
        """

        # create source and target masks
        src_mask = self.make_src_mask(src)  # (batch_size, 1, 1, src_seq_length)
        trg_mask = self.make_trg_mask(trg)  # (batch_size, 1, trg_seq_length, trg_seq_length)

        # push the src through the encoder layers
        src = self.encoder(self.src_embed(src), src_mask)  # (batch_size, src_seq_length, d_model)

        # decoder output and attention probabilities
        output = self.decoder(self.trg_embed(trg), src, trg_mask, src_mask)

        return output


Unlike a version that builds embeddings and layer stacks inside the `Transformer` class itself, here
`Transformer` just wires together modules that are already built (`encoder`, `decoder`, `src_embed`,
`trg_embed`) — construction happens in `helper.make_model` (Part 3). There's also no separate
`encode()`/`decode()` API; `forward()` does both in one call, and mask-building lives here as two methods,
`make_src_mask` and `make_trg_mask`, rather than as standalone functions in the data pipeline.

# Part 2: The data pipeline

This project uses spaCy tokenizers plus `torchtext.vocab.build_vocab_from_iterator` and a plain PyTorch
`DataLoader` with a custom `collate_fn` — not the legacy `torchtext.data.Field` / `BucketIterator` / `Example`
API (which is deprecated and removed in modern torchtext).

Note: in the real project these functions live in two separate files, `translation.py` and `helper.py`, that
actually import each other — `translation.py` calls `helper.load_parallel(...)`, and `helper.py` calls
`translation.load_tokenizers(...)`/`translation.load_vocab(...)`. That's a circular import at the *file*
level, and it only works in the real project because neither module touches the other's names until a
function is actually called, not at import time. There's no such cycle at the *function* level, though, so
this notebook flattens both files into one namespace and orders every function by what it actually calls:
leaf helpers first (`load_parallel`, `load_tokenizers`, `tokenize`, `yield_tokens`), then vocabulary building
(`build_vocabulary`, `load_vocab`), then batching (`data_process`, `generate_batch`) — nothing below is called
before it's declared.


In [ ]:
def load_parallel(de_path, en_path):
    """Yield (german, english) sentence pairs from two aligned plain-text files."""
    with open(de_path, encoding="utf-8") as f_de, open(en_path, encoding="utf-8") as f_en:
        for de, en in zip(f_de, f_en):
            yield de.strip(), en.strip()


def load_tokenizers():
    """
      Load the German and English tokenizers provided by spaCy.
      Returns:
          spacy_de:     German tokenizer
          spacy_en:     English tokenizer
    """
    try:
        spacy_de = spacy.load("de_core_news_sm")
    except OSError:
        os.system("python -m spacy download de_core_news_sm")
        spacy_de = spacy.load("de_core_news_sm")
    try:
        spacy_en = spacy.load("en_core_web_sm")
    except OSError:
        os.system("python -m spacy download en_core_web_sm")
        spacy_en = spacy.load("en_core_web_sm")

    print("Loaded English and German tokenizers.")
    return spacy_de, spacy_en


def tokenize(text: str, tokenizer):
    """
      Split a string into its tokens using the provided tokenizer.
      Args:
          text:         string
          tokenizer:    tokenizer for the language
      Returns:
          tokenized list of strings
          print(tokenize("A Man with an Orange Hat.", spacy_en))
          # ['a', 'man', 'with', 'an', 'orange', 'hat', '.']
    """
    return [tok.text.lower() for tok in tokenizer.tokenizer(text)]


def yield_tokens(data_iter, tokenizer, index: int):
    """
      Return the tokens for the appropriate language.
      Args:
          data_iter:    German/English sentence-pair tuples
          tokenizer:    tokenizer for the language
          index:        index of the language in the tuple | (de=0, en=1)
      Yields:
          sequences based on index
    """
    for from_tuple in data_iter:
        yield tokenizer(from_tuple[index])


With those in place, build the (cached) vocabulary out of them.


In [ ]:
def build_vocabulary(spacy_de, spacy_en, min_freq: int = 2):
    global vocab_src, vocab_trg

    def tokenize_de(text: str):
        return tokenize(text, spacy_de)

    def tokenize_en(text: str):
        return tokenize(text, spacy_en)

    print("Building German Vocabulary...")

    # load train, val, and test data pipelines
    train = list(load_parallel("data/train.de", "data/train.en"))
    val = list(load_parallel("data/val.de", "data/val.en"))
    test = list(load_parallel("data/test2016.de", "data/test2016.en"))

    # generate source vocabulary
    try:
        vocab_src = build_vocab_from_iterator(
            yield_tokens(train + val + test, tokenize_de, index=0),  # tokens for each German sentence (index 0)
            min_freq=min_freq,
            specials=["<bos>", "<eos>", "<pad>", "<unk>"],
        )
    except (ValueError, IndexError):
        print('Error happened!')

    print("Building English Vocabulary...")

    # generate target vocabulary
    try:
        vocab_trg = build_vocab_from_iterator(
            yield_tokens(train + val + test, tokenize_en, index=1),  # tokens for each English sentence (index 1)
            min_freq=2,
            specials=["<bos>", "<eos>", "<pad>", "<unk>"],
        )
    except (ValueError, IndexError):
        print('Error happened!')

    # set default token for out-of-vocabulary words (OOV)
    vocab_src.set_default_index(vocab_src["<unk>"])
    vocab_trg.set_default_index(vocab_trg["<unk>"])

    return vocab_src, vocab_trg


def load_vocab(spacy_de, spacy_en, min_freq: int = 2):
    """
      Args:
          spacy_de:     German tokenizer
          spacy_en:     English tokenizer
          min_freq:     minimum frequency needed to include a word in the vocabulary
      Returns:
          vocab_src:    German vocabulary
          vocab_trg:    English vocabulary
    """
    if not os.path.exists("vocab.pt"):
        # build the German/English vocabulary if it does not exist
        vocab_src, vocab_trg = build_vocabulary(spacy_de, spacy_en, min_freq)
        # save it to a file
        torch.save((vocab_src, vocab_trg), "vocab.pt")
    else:
        # load the vocab if it exists
        vocab_src, vocab_trg = torch.load("vocab.pt")

    print("Finished.\nVocabulary sizes:")
    print("\tSource:", len(vocab_src))
    print("\tTarget:", len(vocab_trg))
    return vocab_src, vocab_trg


The only caching this project does is of the **vocabulary** — `load_vocab` builds it once and saves it to
`vocab.pt`, reloading from disk on subsequent runs. There's no dataset-tokenization cache like some reference
implementations use; the parallel corpus is small enough that `data_process` (next cell) just re-tokenizes
every run.

Now the functions that actually turn raw sentence pairs into padded index tensors for a batch — these depend
on `load_tokenizers` and `load_vocab` above.


In [ ]:
def data_process(raw_data):
  """
    Process raw sentences by tokenizing and converting to integers based on
    the vocabulary.
    Args:
        raw_data:     German-English sentence pairs
    Returns:
        data:         tokenized data converted to index based on vocabulary
  """
  data = []
  spacy_de, spacy_en = load_tokenizers()
  vocab_src, vocab_trg = load_vocab(spacy_de, spacy_en)
  # loop through each sentence pair
  for (raw_de, raw_en) in raw_data:
    # tokenize the sentence and convert each word to an integers
    de_tensor_ = torch.tensor([vocab_src[token.text.lower()] for token in spacy_de.tokenizer(raw_de)], dtype=torch.long)
    en_tensor_ = torch.tensor([vocab_trg[token.text.lower()] for token in spacy_en.tokenizer(raw_en)], dtype=torch.long)

    # append tensor representations
    data.append((de_tensor_, en_tensor_))
  return data


def generate_batch(data_batch):
    """
      Process indexed-sequences by adding <bos>, <eos>, and <pad> tokens.
      Args:
          data_batch:     German-English indexed-sentence pairs
      Returns:
          two batches:    one for German and one for English
    """
    de_batch, en_batch = [], []
    spacy_de, spacy_en = load_tokenizers()
    vocab_src, vocab_trg = load_vocab(spacy_de, spacy_en)
    BOS_IDX = vocab_trg['<bos>']
    EOS_IDX = vocab_trg['<eos>']
    PAD_IDX = vocab_trg['<pad>']
    MAX_PADDING = 20
    BATCH_SIZE = 128

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # for each sentence
    for (de_item, en_item) in data_batch:
        # add <bos> and <eos> indices before and after the sentence
        de_temp = torch.cat([torch.tensor([BOS_IDX]), de_item, torch.tensor([EOS_IDX])], dim=0).to(device)
        en_temp = torch.cat([torch.tensor([BOS_IDX]), en_item, torch.tensor([EOS_IDX])], dim=0).to(device)

        # add padding
        de_batch.append(pad(de_temp, (0, MAX_PADDING - len(de_temp)), value=PAD_IDX))

        # add padding
        en_batch.append(pad(en_temp, (0, MAX_PADDING - len(en_temp)), value=PAD_IDX))

    return torch.stack(de_batch), torch.stack(en_batch)


`generate_batch` is the `collate_fn` passed to the `DataLoader`s (Part 4). Note `MAX_PADDING = 20` is
hardcoded to a fixed length here (sentences longer than 20 tokens after `<bos>`/`<eos>` would make `pad`'s
negative-padding argument error out), matching the `MAX_PADDING` constant in Cell 2.

# Part 3: Training and translation utilities

With the model (Part 1) and data pipeline (Part 2) ready, here are the functions used directly in the
training loop and at inference time.


In [ ]:
def make_model(device,
               src_vocab,
               trg_vocab,
               n_layers: int = 3,
               d_model: int = 512,
               d_ffn: int = 2048,
               n_heads: int = 8,
               dropout: float = 0.1,
               max_length: int = 5000):
    """
      Construct a model when provided parameters.
      Args:
          src_vocab:    source vocabulary
          trg_vocab:    target vocabulary
          n_layers:     number of stacked layers in the encoder and in the decoder
          d_model:      dimension of embeddings
          d_ffn:        dimension of feed-forward network
          n_heads:      number of heads
          dropout:      probability of dropout occurring
          max_length:   maximum sequence length for positional encodings
      Returns:
          Transformer model based on hyperparameters
      """
    # create source embedding matrix
    src_embed = Embeddings(len(src_vocab), d_model)

    # create target embedding matrix
    trg_embed = Embeddings(len(trg_vocab), d_model)

    # create a positional encoding matrix
    pos_enc = PositionalEncoding(d_model, dropout, max_length)

    # create the encoder
    encoder = Encoder(d_model, n_layers, n_heads, d_ffn, dropout)

    # create the decoder
    decoder = Decoder(len(trg_vocab), d_model, n_layers, n_heads, d_ffn, dropout)

    # create the Transformer model
    model = Transformer(
        encoder,
        decoder,
        nn.Sequential(src_embed, pos_enc),
        nn.Sequential(trg_embed, pos_enc),
        src_pad_idx=src_vocab.get_stoi()["<pad>"],
        trg_pad_idx=trg_vocab.get_stoi()["<pad>"],
        device=device
    )

    # initialize parameters with Xavier/Glorot
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)

    return model


Both `src_embed` and `trg_embed` reuse the *same* `pos_enc` module instance (only the token-embedding lookup
tables differ) — one `PositionalEncoding` buffer is shared between the two `nn.Sequential` stacks rather than
each getting its own copy.

Training and evaluation are a standard supervised loop: teacher-forcing on the shifted target sequence,
`nn.CrossEntropyLoss` (no label smoothing), and gradient clipping. There's no learning-rate warm-up — the
optimizer is plain `Adam` at a fixed `LEARNING_RATE` (Cell 2).


In [ ]:
def train(model, iterator, optimizer, criterion, clip):
    """
      Train the model on the given data.
      Args:
          model:        Transformer model to be trained
          iterator:     data to be trained on
          optimizer:    optimizer for updating parameters
          criterion:    loss function for updating parameters
          clip:         value to help prevent exploding gradients
      Returns:
          loss for the epoch
    """
    model.train()

    epoch_loss = 0

    for i, batch in enumerate(iterator):
        src, trg = batch

        optimizer.zero_grad()

        # logits for each output
        logits = model(src, trg[:, :-1])

        # expected output
        expected_output = trg[:, 1:]

        # calculate the loss
        loss = criterion(logits.contiguous().view(-1, logits.shape[-1]),
                         expected_output.contiguous().view(-1))

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)


def evaluate(model, iterator, criterion):
    """
      Evaluate the model on the given data.
      Args:
          model:        Transformer model to be trained
          iterator:     data to be evaluated
          criterion:    loss function for assessing outputs
      Returns:
          loss for the data
    """
    model.eval()

    epoch_loss = 0

    with torch.no_grad():
        for i, batch in enumerate(iterator):
            src, trg = batch

            logits = model(src, trg[:, :-1])
            expected_output = trg[:, 1:]

            loss = criterion(logits.contiguous().view(-1, logits.shape[-1]),
                             expected_output.contiguous().view(-1))

            epoch_loss += loss.item()

    return epoch_loss / len(iterator)


def epoch_time(start_time, end_time):
  elapsed_time = end_time - start_time
  elapsed_mins = int(elapsed_time / 60)
  elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
  return elapsed_mins, elapsed_secs


Translation is **greedy, single-sentence** decoding — no beam search, no batching at inference time. Starting
from `<bos>`, the model is re-run on the growing target prefix until it emits `<eos>` (or hits `max_length`),
one new token at a time.


In [ ]:
def translate_sentence(sentence, model, device, max_length=50):
    """
      Translate a German sentence to its English equivalent.

      Args:
          sentence:     German sentence to be translated to English; list or str
          model:        Transformer model used for translation
          device:       device to perform translation on
          max_length:   maximum token length for translation

      Returns:
          src:                  return the tokenized input
          trg_input:            return the input to the decoder before the final output
          trg_output:           return the final translation, shifted right
          attn_probs:           return the attention scores for the decoder heads
          masked_attn_probs:    return the masked attention scores for the decoder heads
    """
    model.eval()
    spacy_de, spacy_en = load_tokenizers()
    vocab_src, vocab_trg = load_vocab(spacy_de, spacy_en)

    # tokenize and index the provided string
    if isinstance(sentence, str):
        src = ['<bos>'] + [token.text.lower() for token in spacy_de(sentence)] + ['<eos>']
    else:
        src = ['<bos>'] + sentence + ['<eos>']

    # convert to integers
    src_indexes = [vocab_src[token] for token in src]

    # convert list to tensor
    src_tensor = torch.tensor(src_indexes).int().unsqueeze(0).to(device)

    # set <bos> token for target generation
    trg_indexes = [vocab_trg.get_stoi()['<bos>']]

    # generate new tokens
    for i in range(max_length):
        trg_tensor = torch.tensor(trg_indexes).int().unsqueeze(0).to(device)

        with torch.no_grad():
            # generate the logits
            logits = model.forward(src_tensor, trg_tensor)

            # select the newly predicted token
            pred_token = logits.argmax(2)[:, -1].item()

            # if <eos> token or max length, stop generating
            if pred_token == vocab_trg.get_stoi()['<eos>'] or i == (max_length - 1):
                trg_input = vocab_trg.lookup_tokens(trg_indexes)
                trg_output = vocab_trg.lookup_tokens(logits.argmax(2).squeeze(0).tolist())

                return src, trg_input, trg_output, model.decoder.attn_probs, model.decoder.masked_attn_probs
            else:
                trg_indexes.append(pred_token)
    return None


`model.decoder.attn_probs` holds the last decoder layer's **cross**-attention weights and
`model.decoder.masked_attn_probs` holds its **self**-attention weights — both are set inside `Decoder.forward`
(Part 1), which is why `DecoderLayer.forward` returns both `attn_probs` and `masked_attn_probs` rather than
just one.

# Part 4: The training script

This is what `main.py` actually runs — hardcoded hyperparameters (no `argparse` CLI), loading the parallel
corpus files, building `DataLoader`s, constructing the model, and the epoch loop.


In [ ]:
N_EPOCHS = 10
CLIP = 1

best_valid_loss = float('inf')
MAX_PADDING = 20
BATCH_SIZE = 128

train_data_raw = list(load_parallel("data/train.de", "data/train.en"))
val_data_raw = list(load_parallel("data/val.de", "data/val.en"))
test_data_raw = list(load_parallel("data/test2016.de", "data/test2016.en"))

# processed data
train_data = data_process(train_data_raw)
val_data = data_process(val_data_raw)
test_data = data_process(test_data_raw)

train_iter = DataLoader(to_map_style_dataset(train_data), batch_size=BATCH_SIZE,
                        shuffle=True, drop_last=True, collate_fn=generate_batch)

valid_iter = DataLoader(to_map_style_dataset(val_data), batch_size=BATCH_SIZE,
                        shuffle=True, drop_last=True, collate_fn=generate_batch)

test_iter = DataLoader(to_map_style_dataset(test_data), batch_size=BATCH_SIZE,
                       shuffle=True, drop_last=True, collate_fn=generate_batch)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

spacy_de, spacy_en = load_tokenizers()
vocab_src, vocab_trg = load_vocab(spacy_de, spacy_en)

BOS_IDX = vocab_trg['<bos>']
EOS_IDX = vocab_trg['<eos>']
PAD_IDX = vocab_trg['<pad>']

model = make_model(device, vocab_src, vocab_trg,
                   n_layers=3, n_heads=8, d_model=256,
                   d_ffn=512, max_length=50)
model.to(device)

LEARNING_RATE = 0.0005

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


`model.to(device)` moves the model to whichever device was picked (`cuda` if available, else `cpu`) — matching
the device that `generate_batch` already puts every batch tensor on.


In [ ]:
for epoch in range(N_EPOCHS):

    start_time = time.time()

    # calculate the train loss and update the parameters
    train_loss = train(model, train_iter, optimizer, criterion, CLIP)

    # calculate the loss on the validation set
    valid_loss = evaluate(model, valid_iter, criterion)

    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    # save the model when it performs better than the previous run
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'transformer-model.pt')

    print(f'Epoch: {epoch + 1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

# load the best-scoring weights back before doing anything else with the model
model.load_state_dict(torch.load('transformer-model.pt'))


Only the best-so-far validation checkpoint is ever kept, overwriting the same `transformer-model.pt` file
each time — there's no per-epoch checkpoint history like `transformer_ckpt_epoch_N.pth` naming schemes some
reference implementations use, and no test-set evaluation call wired up (it's present but commented out in
`main.py`).

# Part 5: Translating a sentence and visualizing attention

Finally, translate a single hardcoded example sentence and plot the decoder's masked self-attention.


In [ ]:
def display_attention(
        sentence: list,
        translation: list,
        attention: Tensor,
        n_heads: int = 8,
        n_rows: int = 4,
        n_cols: int = 2):
    """
      Display the attention matrix for each head of a sequence.
      Args:
          sentence:     German sentence to be translated to English; list
          translation:  English sentence predicted by the model
          attention:    attention scores for the heads
          n_heads:      number of heads
          n_rows:       number of rows
          n_cols:       number of columns
    """
    assert n_rows * n_cols == n_heads

    fig = plt.figure(figsize=(15, 25))

    for i in range(n_heads):
        ax = fig.add_subplot(n_rows, n_cols, i + 1)

        # select the respective head and make it a numpy array for plotting
        _attention = attention.squeeze(0)[i, :, :].cpu().detach().numpy()

        # plot the matrix
        cax = ax.matshow(_attention, cmap='bone')

        ax.tick_params(labelsize=12)

        ax.set_xticks(range(len(sentence)))
        ax.set_yticks(range(len(translation)))

        if isinstance(sentence[0], str):
            ax.set_xticklabels([t.lower() for t in sentence], rotation=45)
            ax.set_yticklabels(translation)
        elif isinstance(sentence[0], int):
            ax.set_xticklabels(sentence)
            ax.set_yticklabels(translation)

    plt.show()


`display_attention` uses plain `matplotlib.matshow` in an 8-subplot (4x2) grid — one subplot per attention
head — rather than a `seaborn` heatmap grid. Because `Decoder.forward` only caches the last layer's
`attn_probs` (Part 1), whatever is passed in here is necessarily the final decoder layer's attention, not a
per-layer breakdown.


In [ ]:
# 'a woman with a large purse is walking by a gate'
src = ['eine', 'frau', 'mit', 'einer', 'großen', 'geldbörse', 'geht', 'an', 'einem', 'tor', 'vorbei', '.']

src, trg_input, trg_output, attn_probs, masked_attn_probs = translate_sentence(src, model, device)

print(f'source = {src}')
print(f'target input = {trg_input}')
print(f'target output = {trg_output}')

display_attention(trg_input, trg_input, masked_attn_probs)


That's the whole pipeline: `Embeddings` + `PositionalEncoding` → `Encoder`/`Decoder` stacks of
`MultiHeadAttention` + `PositionwiseFeedForward` → `Transformer.forward` producing logits → `train`/`evaluate`
with plain `CrossEntropyLoss` and fixed-LR `Adam` → greedy `translate_sentence` → `display_attention`.

## Additional resources

For conceptual background beyond this repo's specific code, [Jay Alammar's *The Illustrated
Transformer*](https://jalammar.github.io/illustrated-transformer/) is a good complement to the [original
paper](https://arxiv.org/pdf/1706.03762.pdf).
